In [1]:
import os
import jsonlines
import pandas as pd

os.chdir('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data')
acc_info = []
with open('assembly_data_report.jsonl', 'r') as file:
    for line in jsonlines.Reader(file):
        acc_info.append(pd.DataFrame([{'accession': line['accession'], 'organismName': line['organism']['organismName'], 'taxId': line['organism']['taxId']}]))

acc_info = pd.concat(acc_info, ignore_index=True)
print(len(acc_info))

51772


In [2]:
from Bio import SeqIO
from tqdm import tqdm
import multiprocessing

def get_acc_info(acc_n, que):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    seq_record = SeqIO.parse(handle, 'genbank')
    temp_statistic = {'accession': acc_n, 'chromosome number':0, 'chr_max_size':None, 'chr_min_size':None, 'chr_sum_size':None, 'plasmid number':0, 'pla_max_size':None, 'pla_min_size':None, 'total number':0, 'chromosome contigs':None, 'plasmid contigs':None,}
    chr_accs = {}
    pla_accs = {}
    for record in seq_record:
        temp_statistic['total number'] += 1
        if 'chromosome' in record.description:
            temp_statistic['chromosome number'] += 1
            chr_accs[record.id] = len(record.seq)
            if temp_statistic['chr_sum_size'] == None:
                temp_statistic['chr_sum_size'] = len(record.seq)
            else:
                temp_statistic['chr_sum_size'] += len(record.seq)
            if temp_statistic['chr_max_size'] == None or len(record.seq) > temp_statistic['chr_max_size']:
                temp_statistic['chr_max_size'] = len(record.seq)
            if temp_statistic['chr_min_size'] == None or len(record.seq) < temp_statistic['chr_min_size']:
                temp_statistic['chr_min_size'] = len(record.seq)
        elif 'plasmid' in record.description:
            temp_statistic['plasmid number'] += 1
            pla_accs[record.id] = len(record.seq)
            if temp_statistic['pla_max_size'] == None or len(record.seq) > temp_statistic['pla_max_size']:
                temp_statistic['pla_max_size'] = len(record.seq)
            if temp_statistic['pla_min_size'] == None or len(record.seq) < temp_statistic['pla_min_size']:
                temp_statistic['pla_min_size'] = len(record.seq)
        else:
            temp_statistic['chromosome number'] += 1
            chr_accs[record.id] = len(record.seq)
            if temp_statistic['chr_sum_size'] == None:
                temp_statistic['chr_sum_size'] = len(record.seq)
            else:
                temp_statistic['chr_sum_size'] += len(record.seq)
            if temp_statistic['chr_max_size'] == None or len(record.seq) > temp_statistic['chr_max_size']:
                temp_statistic['chr_max_size'] = len(record.seq)
            if temp_statistic['chr_min_size'] == None or len(record.seq) < temp_statistic['chr_min_size']:
                temp_statistic['chr_min_size'] = len(record.seq)
    temp_statistic['chromosome contigs'] = str(chr_accs)
    temp_statistic['plasmid contigs'] = str(pla_accs)
    que.put(temp_statistic)

manager = multiprocessing.Manager()
que = manager.Queue()
par = 32
acc_list = list(acc_info['accession'])
tot = len(acc_list)
pool = multiprocessing.Pool(par)

for acc_n in acc_list:
    pool.apply_async(get_acc_info, (acc_n, que))

pool.close()

with tqdm(total = len(acc_list), desc='Program', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
    columns = ['accession', 'chromosome number', 'chr_max_size', 'chr_min_size', 'chr_sum_size', 'plasmid number', 'pla_max_size', 'pla_min_size', 'total number', 'chromosome contigs', 'plasmid contigs']
    all_statistics = pd.DataFrame(columns = columns)
    count = 0
    while True:
        if not que.empty():
            temp_statistic = que.get(True)
            all_statistics = pd.concat([all_statistics, pd.DataFrame([temp_statistic])], ignore_index=True)
            count += 1
            pbar.update(1)
            if count == tot:
                break
        else:
            continue

pool.join()
merge_table = pd.merge(acc_info, all_statistics, on='accession', how='left')
data_dir = '/active-data/analysis_results/chr_pla'
os.makedirs(data_dir, exist_ok=True)
os.chdir(data_dir)
merge_table.to_csv('genome_chr-pla_statistics.csv', index=False)

Program: 100%|██████████████████████████████████████████████████| 51.8k/51.8k [16:34<00:00, 52.1B/s]


In [3]:
import os
import json
import subprocess

def tax_classification(taxId, que):
    result = subprocess.run(["datasets", "summary", "taxonomy", "taxon", str(taxId)], capture_output=True, text=True)
    records = []
    for line in result.stdout.strip().splitlines():
        if line:
            records.append(json.loads(line))
    try:
        tax_class = records[0]['reports'][0]['taxonomy']['classification']
    except:
        tax_class = {}
    tax_class['taxId'] = taxId
    que.put(tax_class)

taxons = merge_table['taxId'].value_counts()
matched_taxons = []
taxon_class = []
time_count = 0

while len(matched_taxons) < len(taxons):
    time_count += 1
    manager = multiprocessing.Manager()
    que = manager.Queue()
    par = 64
    tot = 0
    pool = multiprocessing.Pool(par)
    for taxId in taxons.index:
        if taxId in matched_taxons:
            continue
        else:
            pool.apply_async(tax_classification, (taxId, que))
            tot += 1

    pool.close()
    with tqdm(total = tot, desc=f'Program-{time_count}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        count = 0
        while True:
            if not que.empty():
                temp_taxon = que.get(True)
                if len(temp_taxon) == 1:
                    pass
                else:
                    taxon_class.append(pd.DataFrame([temp_taxon]))
                    matched_taxons.append(temp_taxon['taxId'])
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
                
    pool.join()

taxon_class = pd.concat(taxon_class, ignore_index=True)

Program-1: 100%|██████████████████████████████████████████████| 16.3k/16.3k [2:41:10<00:00, 1.69B/s]
Program-2: 100%|██████████████████████████████████████████████| 6.91k/6.91k [1:16:00<00:00, 1.52B/s]
Program-3: 100%|████████████████████████████████████████████████| 2.23k/2.23k [24:47<00:00, 1.50B/s]
Program-5: 100%|████████████████████████████████████████████████████| 240/240 [02:44<00:00, 1.46B/s]
Program-6: 100%|██████████████████████████████████████████████████| 73.0/73.0 [01:21<00:00, 1.12s/B]
Program-7: 100%|██████████████████████████████████████████████████| 13.0/13.0 [00:21<00:00, 1.67s/B]


In [4]:
taxon_merge_table = pd.merge(merge_table, taxon_class, on='taxId', how='left')
os.chdir(data_dir)
taxon_merge_table.to_csv('genome_chr-pla_statistics.csv', index=False)